In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.

In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [4]:
train_data = datasets.IMDB(split='train')
test_data = datasets.IMDB(split='test')

In [5]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [6]:
tokenizer = get_tokenizer("basic_english")

In [7]:
# проверяем какой столбец первым приходит(класс или данные)
for i in list(train_data)[:5]: # берём только 5 первых
    print(i)

(1, 'I guess this would be a great movie for a true believer in organized Christian Dogma, but for anyone with an open mind who believes in free will, rational thinking, the separation of Church & State and GOOD Science Fiction it is a terrible joke!<br /><br />There are some well known actors who were either badly in need of work or had a need to share their personal beliefs with the rest of us heathens.<br /><br />I WAS entertained by this movie in the same way I was entertained by "Reefer Madness." That movie attempted to teach drug education by scare tactics the same way this movie tries to teach "Christian" principles with the threat of hell and misery for otherwise good people who don\'t share their interpretations of our world.<br /><br />It had me howling with laughter and at the same time scared me to realize how many people actually believe that our society should revert to the good old days of the 19th century!')
(1, 'The director, outfitted in chains and leather, warned the

In [8]:
def text_to_token(data):
    for label, text in data:
        yield tokenizer(text)

In [9]:
vocab = build_vocab_from_iterator(text_to_token(train_data), specials=["<unk>"])

In [10]:
vocab.set_default_index(vocab["<unk>"])

In [11]:
def change_text(x):
    return [vocab[i] for i in tokenizer(x)]

change_label = lambda name : 1 if name == 'positive' else 0

In [12]:
def collate_batch(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(change_label(label))
        texts.append(torch.tensor(change_text(text), dtype=torch.int64))
    labels = torch.tensor(labels, dtype=torch.int64)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    return texts, labels

In [13]:
train_dataloader = DataLoader(list(train_data), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_data), batch_size=32, collate_fn=collate_batch)

In [14]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
model = SentimentModel(len(vocab)).to(device)

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
for epoch in range(15):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 3.32
Эпоха: 2 - Потери: 0.01
Эпоха: 3 - Потери: 0.0
Эпоха: 4 - Потери: 0.0
Эпоха: 5 - Потери: 0.0
Эпоха: 6 - Потери: 0.0
Эпоха: 7 - Потери: 0.0
Эпоха: 8 - Потери: 0.0
Эпоха: 9 - Потери: 0.0
Эпоха: 10 - Потери: 0.0
Эпоха: 11 - Потери: 0.0
Эпоха: 12 - Потери: 0.0
Эпоха: 13 - Потери: 0.0
Эпоха: 14 - Потери: 0.0
Эпоха: 15 - Потери: 0.0


In [19]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 100.00%


In [20]:
from google.colab import files

torch.save(vocab, 'vocab_IMDB _DatasetOf_50K _MovieReviews.pth')
torch.save(model.state_dict(), 'model_IMDB_DatasetOf_50K_MovieReviews.pth')

files.download('vocab_IMDB _DatasetOf_50K _MovieReviews.pth')
files.download('model_IMDB_DatasetOf_50K_MovieReviews.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>